# Notebook 01: OncoLung60K Dataset Overview

This notebook walks through:
1. Loading the patient-wise k-fold split CSV
2. Verifying no patient leakage between folds
3. Visualising class distribution per fold
4. Inspecting sample patches


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.utils.splits import verify_no_patient_leakage, summarise_splits

## 1. Load splits

In [ ]:
df = pd.read_csv('../splits/oncolung60k_5fold.csv')
print(f'Total patches: {len(df):,}')
print(f'Patients:      {df["patient_id"].nunique()}')
print(f'Classes:       {sorted(df["label"].unique())}')
df.head()

## 2. Verify no patient leakage

In [ ]:
verify_no_patient_leakage(df)
summary = summarise_splits(df)

## 3. Visualise class distribution

In [ ]:
CLASS_NAMES = ['Adeno', 'SCC', 'SCLC', 'Normal']
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

patient_counts = df.groupby('patient_id')['label'].first().value_counts().sort_index()
axes[0].bar([CLASS_NAMES[i] for i in patient_counts.index], patient_counts.values)
axes[0].set_title('Patients per Class'); axes[0].set_ylabel('# Patients')
for i, v in enumerate(patient_counts.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

patch_counts = df['label'].value_counts().sort_index()
axes[1].bar([CLASS_NAMES[i] for i in patch_counts.index], patch_counts.values)
axes[1].set_title('Patches per Class'); axes[1].set_ylabel('# Patches')
for i, v in enumerate(patch_counts.values):
    axes[1].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout(); plt.show()

## 4. Inspect sample patches

(Requires the dataset to be downloaded to `data/oncolung60k/`)

In [ ]:
DATA_ROOT = Path('../data/oncolung60k')  # adjust as needed

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for c in range(4):
    samples = df[df['label'] == c].sample(4, random_state=c)
    for i, (_, row) in enumerate(samples.iterrows()):
        path = DATA_ROOT / row['filepath']
        if not path.exists():
            axes[c, i].text(0.5, 0.5, f'Image not found:\n{row["filepath"]}',
                            ha='center', va='center')
            axes[c, i].set_xticks([]); axes[c, i].set_yticks([])
            continue
        img = Image.open(path)
        axes[c, i].imshow(img)
        axes[c, i].set_title(f'{CLASS_NAMES[c]} - {row["patient_id"]}', fontsize=10)
        axes[c, i].axis('off')
plt.suptitle('Sample Patches per Class', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()